# MiddleWares in Langchain

In [31]:
from langchain.agents import create_agent 
from langchain_mistralai import ChatMistralAI 
from langchain.tools import tool 
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint

from dotenv import load_dotenv 
load_dotenv()

True

In [12]:
agent = create_agent(
    model="mistral-medium-latest",
    middleware=[
        SummarizationMiddleware(
            model="mistral-small-latest",
            trigger=("tokens", 1000),
            keep=("messages", 2),
        ),
    ],
    checkpointer= InMemorySaver()
)

In [19]:
user_input = input()
response = agent.invoke(
    {
        "messages" : {"role":"user","content":user_input}
    },
    config = {"configurable" : {"thread_id":"1"}}
)
print(response['messages'][-1].content)

Here’s the exact poem I wrote earlier for **Rohit Sharma**—*"The Hitman’s Symphony"*—unchanged:

---

### **🏏 *The Hitman’s Symphony* 🏏**

Beneath the floodlights’ golden gaze,
A shadow steps with quiet blaze—
Not storm nor quake, but *timing’s* art,
A maestro with a willow’s heart.

The crowd inhales—*the bowler runs*,
A spell of pace, of grunts and guns.
But *he* stands still, a sculpted calm,
While time itself bends to his palm.

**A flick!** The bat kisses the sphere,
A sound like silk—yet *fear* is near.
For boundaries *melt* where he commands,
A six!—no fence, no outstretched hands.

*Pull! Cut! Drive!* The field’s a stage,
His strokes compose a *hitman’s* rage.
No mercy left, no run too steep,
The bowlers weep, the scorers weep.

Yet in his eyes—no fire, no ire,
Just *numbers* climbing, cool and higher.
A hundred falls like morning dew,
Another chase *dismantled* anew.

So raise the cup, let anthems ring,
For kings may fall, but *he* is king.
Not just a man, but *cricket’s* cres

In [32]:
pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user initially requested a poem about cricket (Rohit Sharma), then shifted to Rolls-Royce, and later asked about the fastest bird. The latest request is about owls.\n\n## SUMMARY\n1. A cricket-themed poem ("The Hitman’s Symphony") was created for Rohit Sharma, with options for further customization.\n2. Detailed information about Rolls-Royce was provided, including history, models, features, and future plans.\n3. The user asked about the fastest bird (Peregrine Falcon) and now wants information about owls.\n\n## ARTIFACTS\n- Poem: "The Hitman’s Symphony" (created and shared).\n- Rolls-Royce information (shared in the conversation).\n- Fastest bird information (Peregrine Falcon details).\n\n## NEXT STEPS\n- Provide information about owls.\n- If the user revisits prior topics (cricket/poem, Rolls-Royce, or fastest bird), address accordingly.', additional_kwargs={'lc_source': 'summ

In [92]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool 
def read_email_tool(email_id: str) -> str:
    """function to read an email by its ID."""
    return f"Email content for ID: {email_id}"
@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """ function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="mistral-medium-latest",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [93]:
response = agent.invoke(
    {
        "messages" : [{"role":"user","content":"send an email to dhairya using this user_id : dhairyavaghela@openai.com"}]
    },
    config = {"configurable":{"thread_id":"1"}}
)
response

{'messages': [HumanMessage(content='send an email to dhairya using this user_id : dhairyavaghela@openai.com', additional_kwargs={}, response_metadata={}, id='c87f7f96-90bb-4c91-a5a4-b5865b1fb3b3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'uaTUKwJs0', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient": "dhairyavaghela@openai.com", "subject": "", "body": ""}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 174, 'total_tokens': 207, 'completion_tokens': 33, 'prompt_tokens_details': {'cached_tokens': 144}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ce103-4089-7bb0-9b68-dc81983ff46a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'dhairyavaghela@openai.com', 'subject': '', 'body': ''}, 'id': 'uaTUKwJs0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 174, 'output_tokens': 33

In [94]:
response = agent.invoke(
    Command(
        resume={
            "decisions":[{
                "type":"edit",
                "edited_action" : {
                    "name" : "send_email_tool",
                    "args" : {
                    "recipient" : "dhairyavaghela12@gmail.com",
                    "subject" : "Invitation from Standford.",
                    "body" : "we Invite you to study at our University"
                }}
                }]}),
    config = {"configurable":{"thread_id":"1"}},
)

In [95]:
response

{'messages': [HumanMessage(content='send an email to dhairya using this user_id : dhairyavaghela@openai.com', additional_kwargs={}, response_metadata={}, id='c87f7f96-90bb-4c91-a5a4-b5865b1fb3b3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'uaTUKwJs0', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient": "dhairyavaghela@openai.com", "subject": "", "body": ""}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 174, 'total_tokens': 207, 'completion_tokens': 33, 'prompt_tokens_details': {'cached_tokens': 144}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ce103-4089-7bb0-9b68-dc81983ff46a-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'dhairyavaghela12@gmail.com', 'subject': 'Invitation from Standford.', 'body': 'we Invite you to study at our University'}, 'id': 'uaTUKwJs0'}], invalid_tool

# Model Call Limit

In [113]:
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver 

agent = create_agent(
    model = "mistral-medium-2508",
    checkpointer= InMemorySaver(),
    middleware= [ModelCallLimitMiddleware(
        thread_limit=3,
        run_limit= 5,
        exit_behavior = "end",
    )],
)

In [117]:
user_prompt = input("User Prompt : ")
response = agent.invoke(
    {
        "messages" : [{"role":"user","content":user_prompt}]
    },
    config = {"configurable":{"thread_id":"1"}}
)
print("AI Response : ",response['messages'][-1].content)

AI Response :  Model call limits exceeded: thread limit (3/3)


In [126]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Hello
================================== Ai Message ==================================

Hello! 😊 How can I help you today? Whether you have a question, need advice, or just want to chat, I'm here for you! Let me know what's on your mind. 🚀

(Or if you're testing me, try asking something fun like *"Tell me a random fact!"* or *"What’s the meaning of life?"* 😄)
================================ Human Message =================================

Tell me a random fact
================================== Ai Message ==================================

Here’s a fun one:

**Honey never spoils!** Archaeologists have found pots of honey in ancient Egyptian tombs that are over **3,000 years old**—and still perfectly edible. Thanks to its low moisture content, high acidity, and natural preservatives, honey is one of the only foods that can last *forever* if stored properly. 🍯⏳

Want another? Or a fact about a specific to

# ToolCallMiddleware

In [155]:
from langchain.agents.middleware import ToolCallLimitMiddleware 

@tool 
def math_function(expression : str) -> str:
    """Perform math operations"""
    return str(eval(expression))

agent = create_agent(
    model = "mistral-medium-latest",
    middleware= [
        ToolCallLimitMiddleware(
            tool_name="math_function",
            thread_limit=2,
            run_limit = 2,
            exit_behavior = "end"
        )
    ],
    tools = [math_function],
    checkpointer=InMemorySaver(),
    system_prompt= "You are an helpful assistant,"
)

In [158]:
user_prompt = input()
response = agent.invoke(
    {
        "messages" : [{"role":"user","content":user_prompt}]
    },
    config = {"configurable":{"thread_id":"1"}}
)
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

'math_function' tool call limit reached: thread limit exceeded (3/2 calls).


In [159]:
for msg in response['messages']:
    msg.pretty_print() 

================================ Human Message =================================

add 4 to 1000
================================== Ai Message ==================================
Tool Calls:
  math_function (bNBfdG2vw)
 Call ID: bNBfdG2vw
  Args:
    expression: 1000 + 4
================================= Tool Message =================================
Name: math_function

1004
================================== Ai Message ==================================

The result of adding **4** to **1000** is **1004**.
================================ Human Message =================================

subtract 500 from previous output
================================== Ai Message ==================================
Tool Calls:
  math_function (2wd0TrCLt)
 Call ID: 2wd0TrCLt
  Args:
    expression: 1004 - 500
================================= Tool Message =================================
Name: math_function

504
================================== Ai Message ==================================

The resul

# Model Fall Back

In [177]:
from langchain.agents.middleware import ModelFallbackMiddleware 

model = ChatMistralAI(
    model = "devstral-2512"
)
agent = create_agent(
    model = model,
    middleware=[
        ModelFallbackMiddleware(
            "mistral-medium-latest"
            )
    ]
)

In [178]:
import base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

base64_image = encode_image(r"d:\AppstoneLab-AI-intern\Concept-Wise\Gen-AI\Streaming\img.jpg")

In [179]:
response = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":[
                    {"type":"text","text":"describe the image."},
                    {
                        "type":"image_url",
                        "image_url":{
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ]
    }
)

response["messages"][-1].pretty_print()

================================== Ai Message ==================================

This image shows two men celebrating a victory in a cricket tournament.

They are both wearing blue sports jerseys with the word "India" and the logo of the Board of Control for Cricket in India (BCCI) on them. One of the men is holding the Indian national flag, while both are wearing medals around their necks. They are also holding a trophy, which appears to be the T20 World Cup trophy, indicating that they have won the tournament. The background shows a stadium filled with spectators, suggesting the event is taking place in front of a live audience. The atmosphere seems celebratory and triumphant.


# PII Detection


In [194]:
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="mistral-medium-latest",
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
    checkpointer = InMemorySaver()
)

In [198]:
response = agent.invoke(
    {
        "messages" : [("user","how do you see my details")]
    },
    config = {"configurable":{"thread_id":"1"}}
)

In [199]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

I **cannot see or access** any of your personal details, including the email address or credit card number you mentioned. Here’s why:

1. **Redaction:** When you included `[REDACTED_EMAIL]` and the credit card number, I treated them as placeholders and did **not** store or process them.
2. **Privacy Policy:** I don’t retain any sensitive information shared in conversations. My responses are generated in real-time without memory of past interactions.
3. **Security:** Never share full financial details (credit card numbers, passwords, etc.) in emails, chats, or unsecured forms. Always use official, encrypted payment portals.

If you’re concerned about privacy, you can:
- Avoid sharing sensitive details in any chat or email.
- Use official company websites/portals for transactions.
- Replace real details with placeholders (like `[Client’s Email]`) when drafting messages.


# Todo list Middleware

In [200]:
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model="mistral-medium-latest",
    middleware=[TodoListMiddleware(
        
    )],
)

In [201]:
response = agent.invoke(
    {
        "messages" : [('user','i want to prepare for exams give me tips')]
    }
)

In [203]:
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Preparing for exams effectively requires a structured approach, good time management, and active learning strategies. Here are some tips to help you prepare efficiently:

---

### **1. Organize Your Study Material**
- **Gather all resources**: Collect your textbooks, notes, lecture slides, and any additional study materials.
- **Create a study plan**: Break down the syllabus into topics and allocate time for each.
- **Prioritize topics**: Focus on high-weightage topics or areas where you feel less confident.

---

### **2. Create a Study Schedule**
- **Set a timeline**: Plan your study sessions leading up to the exam.
- **Divide your time**: Allocate specific time slots for each subject or topic.
- **Include breaks**: Use techniques like the **Pomodoro Technique** (25 minutes of study followed by a 5-minute break).
- **Avoid cramming**: Space out your study sessions to retain information better.

---

###

# LLM Tool selection MiddleWare

In [215]:
from langchain.agents.middleware import LLMToolSelectorMiddleware
agent = create_agent(
    model="mistral-medium-latest",
    tools=[math_function,read_email_tool],
    middleware=[
        LLMToolSelectorMiddleware(
            model="mistral-small-latest",
            max_tools=1,
            always_include=["math_function"],
        ),
    ],
)

In [216]:
response = agent.invoke(
    {
        "messages" : [("user","sum of 100 to 200")]
    }
)
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

The sum of all integers from **100 to 200** is **15,150**.


In [217]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

sum of 100 to 200
================================== Ai Message ==================================
Tool Calls:
  math_function (O8gHYgaYD)
 Call ID: O8gHYgaYD
  Args:
    expression: sum(range(100, 201))
================================= Tool Message =================================
Name: math_function

15150
================================== Ai Message ==================================

The sum of all integers from **100 to 200** is **15,150**.


# LLM Tool Emulator

In [219]:
from langchain.agents.middleware import LLMToolEmulator

@tool 
def get_weather(city : str) -> str:
    """ Returns the weather of a given city."""
    print("Original tool is Executed.")
    return "Weather in {city} is hot."

agent = create_agent(
    model="mistral-medium-2508",
    tools=[get_weather],
    middleware=[
        LLMToolEmulator(model = "mistral-medium-2508"),  
    ],
    checkpointer= InMemorySaver(),
)

In [226]:
response = agent.invoke(
    {
        "messages" : [("user","what is weather in Delhi.?")]
    },
    config = {"configurable":{"thread_id":"1"}}
)
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here’s the latest weather update for **Delhi**:

### **Current Weather**
- **Temperature**: 38°C (Feels like 41°C)
- **Conditions**: Haze with partial smog
- **Humidity**: 42%
- **Wind Speed**: 12 km/h
- **Last Updated**: 2:30 PM (IST)

---

### **Forecast**
- **Today**:
  - **High**: 39°C
  - **Low**: 27°C
  - **Conditions**: Sunny intervals with hazy skies.
  - **Air Quality**: **Unhealthy (AQI: 189)**
  - **Chance of Rain**: 5%

- **Tomorrow**:
  - **High**: 37°C
  - **Low**: 26°C
  - **Conditions**: Partly cloudy with morning fog.
  - **Air Quality**: **Unhealthy for sensitive groups (AQI: 165)**
  - **Chance of Rain**: 10%

---

### **Air Quality Advisory**
- **Level**: High
- **Recommendation**: Limit prolonged outdoor exertion, especially for children, the elderly, and those with respiratory conditions. Use **N95 masks** if outdoors for extended periods.

---
### **Additional Notes**
- Delhi's weat

# Context Editing

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit
from langchain_mistralai import ChatMistralAI 
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

from dotenv import load_dotenv 
load_dotenv()

True

In [4]:
@tool 
def math_function(expression : str) -> str:
    """Perform math operations"""
    return str(eval(expression))

In [51]:
agent = create_agent(
    model = "mistral-medium-2508",
    tools = [math_function],
    middleware= [
        ContextEditingMiddleware(
            edits= [
                ClearToolUsesEdit(
                    trigger = 2,
                    keep = 1,
                    placeholder= "Tools call is Gonna !!"
                )
            ],
        )
    ],
    checkpointer= InMemorySaver()
)

In [55]:
user_input = input()
response = agent.invoke(
    {"messages": ("user", user_input)},
    config={"configurable": {"thread_id": "1"}}
)

print("\n----- STATE -----")
for m in response["messages"]:
    print(type(m).__name__, ":", m.content)


----- STATE -----
HumanMessage : Q-1 : add 9 to 10
AIMessage : 
ToolMessage : 19
AIMessage : The answer is **19**.
HumanMessage : Q-2 : add 99 to 7
AIMessage : 
ToolMessage : 106
AIMessage : The answer is **106**.
HumanMessage : Q-3 : add 54 to 9
AIMessage : 
ToolMessage : 63
AIMessage : The answer is **63**.
HumanMessage : based on the answer of Q-1 add 5 to the answer
AIMessage : 
ToolMessage : 24
AIMessage : The answer is **24**.


In [56]:
response['messages']

[HumanMessage(content='Q-1 : add 9 to 10', additional_kwargs={}, response_metadata={}, id='9c87fa44-e852-482e-acc3-39c01985e718'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6qBSJ36Dj', 'function': {'name': 'math_function', 'arguments': '{"expression": "10 + 9"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 71, 'total_tokens': 86, 'completion_tokens': 15, 'prompt_tokens_details': {'cached_tokens': 32}}, 'model_name': 'mistral-medium-2508', 'model': 'mistral-medium-2508', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ce5b0-ffe5-77a3-8c7d-9cbb7dc09909-0', tool_calls=[{'name': 'math_function', 'args': {'expression': '10 + 9'}, 'id': '6qBSJ36Dj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 15, 'total_tokens': 86}),
 ToolMessage(content='19', name='math_function', id='424374e6-ccd7-4406-85a2-dfc1686bcd33', tool_call_id='6qBSJ36Dj'),
 AIMessage(content='The ans

In [57]:
print(len(response["messages"]))

16


# ShellToolMiddleWare

In [68]:
import os
os.environ["COMSPEC"] = r"C:\Windows\System32\cmd.exe"

In [81]:
from langchain.agents.middleware import ShellToolMiddleware,HostExecutionPolicy 

agent = create_agent(
    model = "mistral-medium-2508",
    middleware=[
        ShellToolMiddleware(
            execution_policy=HostExecutionPolicy(),
        )
    ],
    checkpointer=InMemorySaver()
)


# FileSystemMiddleWare

In [8]:
from langchain.agents.middleware import FilesystemFileSearchMiddleware 
from langchain.agents import create_agent 
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

agent = create_agent(
    model = "mistral-small-latest",
    middleware= [
        FilesystemFileSearchMiddleware(
            root_path = r"d:\AppstoneLab-AI-intern\Concept-Wise\Gen-AI",
            use_ripgrep = True
        )
    ],
    checkpointer = InMemorySaver()
)

In [15]:
response = agent.invoke(
    {
        "messages" : [("user","return the contain of tutorial1.py")]
    },
    config = {"configurable":{"thread_id":"1"}}
)
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here is the content of `tutorial1.py`:

```python
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv() # loading Env

model = ChatOpenRouter(
    model="openai/gpt-oss-120b:free",
    temperature = 1.0
)
print("Model loaded Successfully.")

response = model.invoke("Given an m x n 2D binary grid grid which represents a map of '1's (land) and '0's (water), return the number of islands. in python")
print(response.content)
```
